# Step 7 — Larger SNOWED experiment with a matched decoder

This is the clean experiment. SNOWED supplies 4,334 Sentinel-2 shoreline samples for training and internal evaluation. The 98 rigorously labeled SWED test images remain separate for one external evaluation. ResNet-34, ConvNeXt-Tiny, and DINOv3 ConvNeXt-Tiny all use the same U-Net-style decoder operations and widths.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/hriship618/coastline-image-segmentation.git"
repository = Path("/content/coastline-image-segmentation")
if not repository.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repository)], check=True)
else:
    subprocess.run(["git", "-C", str(repository), "pull", "--ff-only"], check=True)
os.chdir(repository)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
source_directory = str(repository / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)
print(f"Working from: {repository}")

## 1. Download the training dataset
Set `DOWNLOAD_SNOWED` to `True` once. The archive is 6.8 GB. Extraction keeps only the Level-2A image, label, and metadata needed by this project.

In [ ]:
DOWNLOAD_SNOWED = False  # Change to True for the first run.
SNOWED_URL = "https://zenodo.org/records/8112715/files/SNOWED_v02.zip?download=1"
archive_path = Path("/content/data/SNOWED_v02.zip")
snowed_root = Path("/content/data/SNOWED")

if DOWNLOAD_SNOWED:
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["wget", "-c", SNOWED_URL, "-O", str(archive_path)], check=True)
    subprocess.run([
        "unzip", "-q", "-o", str(archive_path),
        "SNOWED/*/sample_2A.npy", "SNOWED/*/label.npy",
        "SNOWED/*/metadata.pkl", "-d", "/content/data"
    ], check=True)
    print(f"Extracted training files under {snowed_root}")
elif not snowed_root.exists():
    raise FileNotFoundError("Set DOWNLOAD_SNOWED = True and run this cell once.")
else:
    print(f"Using existing data under {snowed_root}")

## 2. Verify one real sample
The official archive stores Level-2A reflectance as `[height, width, bands]`. The loader selects red, green, blue, NIR, and SWIR, scales reflectance to 0–1, and aligns the label orientation with the image.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from coastlearn.data import FIVE_BANDS, SnowedDataset, discover_snowed_pairs

snowed_pairs = discover_snowed_pairs(snowed_root)
snowed_dataset = SnowedDataset(snowed_pairs, bands=FIVE_BANDS)
example = snowed_dataset[0]
print("SNOWED pairs:", len(snowed_pairs))
print("image:", tuple(example["image"].shape), "mask:", tuple(example["mask"].shape))

rgb = np.moveaxis(example["image"][:3].numpy(), 0, -1)
rgb = np.clip(rgb * 3.33, 0, 1)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(rgb); axes[0].set_title("SNOWED RGB")
axes[1].imshow(example["mask"], cmap="Blues", vmin=0, vmax=1); axes[1].set_title("Water mask")
for axis in axes: axis.axis("off")
plt.tight_layout(); plt.show()

## 3. Make a geographic split
The metadata contains the original Sentinel MGRS tile. Whole tiles—not individual images—are assigned to train, validation, or internal test, preventing nearby coastline samples from leaking between splits. Metadata is scanned as bytes rather than unpickled.

In [ ]:
import random
import torch
from coastlearn.data import build_snowed_dataloaders, snowed_region_id, split_pairs_by_region

random.seed(7); np.random.seed(7); torch.manual_seed(7)
splits = split_pairs_by_region(
    snowed_pairs, validation_fraction=0.15, test_fraction=0.15, seed=7,
    region_id_function=snowed_region_id,
)
for name, split in (("train", splits.train), ("validation", splits.validation), ("test", splits.test)):
    regions = {snowed_region_id(pair) for pair in split}
    print(name, "images=", len(split), "Sentinel tiles=", len(regions))
loaders = build_snowed_dataloaders(splits, bands=FIVE_BANDS, batch_size=8, num_workers=2)

## 4. Select and train one matched model
Run this section once for each model name. The decoder is held constant. ResNet and ordinary ConvNeXt start from supervised ImageNet weights; DINOv3 ConvNeXt starts from self-supervised DINOv3 weights. The backbone learns slowly while the new decoder learns faster.

In [ ]:
MODEL_NAME = "resnet34_unet"  # Then use convnext_tiny and dinov3_convnext_tiny.
EPOCHS = 10

from google.colab import drive, userdata
from coastlearn.models import build_convnext_tiny, build_dinov3_convnext_tiny, build_resnet34_unet
from coastlearn.training import build_cross_entropy_loss, build_finetuning_optimizer, fit

drive.mount("/content/drive", force_remount=False)
if MODEL_NAME == "dinov3_convnext_tiny":
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    model = build_dinov3_convnext_tiny(in_channels=5, num_classes=2)
elif MODEL_NAME == "convnext_tiny":
    model = build_convnext_tiny(in_channels=5, num_classes=2, pretrained=True)
elif MODEL_NAME == "resnet34_unet":
    model = build_resnet34_unet(in_channels=5, num_classes=2, pretrained=True)
else:
    raise ValueError(f"Unknown model: {MODEL_NAME}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("model:", MODEL_NAME, "device:", device)
loss_function = build_cross_entropy_loss(ignore_index=255).to(device)
optimizer = build_finetuning_optimizer(model, backbone_learning_rate=1e-5, head_learning_rate=1e-3)
checkpoint_path = Path("/content/drive/MyDrive/coastline_segmentation") / f"snowed_{MODEL_NAME}_best.pt"
history = fit(
    model=model, train_loader=loaders["train"], validation_loader=loaders["validation"],
    optimizer=optimizer, loss_function=loss_function, device=device, epochs=EPOCHS,
    checkpoint_path=checkpoint_path, patience=3,
)
print("Saved best checkpoint to:", checkpoint_path)

## 5. Evaluate the best checkpoint
Internal test IoU measures transfer to held-out U.S. Sentinel tiles. The SWED evaluation below is external and optional; it uses the compact 98-pair archive previously saved in Drive and never trains on those images.

In [ ]:
from torch.utils.data import DataLoader
from zipfile import ZipFile
from coastlearn.data import SwedDataset, discover_swed_pairs
from coastlearn.training import evaluate

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"]); model.eval()
internal_metrics = evaluate(model, loaders["test"], loss_function, device)
print("SNOWED held-out test:", internal_metrics)

swed_archive = Path("/content/drive/MyDrive/coastline_segmentation/swed_labeled_pairs.zip")
if swed_archive.exists():
    swed_root = Path("/content/data/swed_external_test")
    swed_root.mkdir(parents=True, exist_ok=True)
    with ZipFile(swed_archive) as archive: archive.extractall(swed_root)
    swed_pairs = discover_swed_pairs(swed_root)
    swed_loader = DataLoader(SwedDataset(swed_pairs, bands=FIVE_BANDS), batch_size=8, shuffle=False, num_workers=2)
    external_metrics = evaluate(model, swed_loader, loss_function, device)
    print("SWED external test:", external_metrics)
else:
    print("SWED archive not found; internal evaluation is complete.")